# 🎵 Waveform Studio — Audio Visualizer Video Generator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mnchrmXD/waveform-studio/blob/main/waveform_studio.ipynb)

Generate studio-grade, audio-reactive visualizer videos directly in Google Colab using **Waveform Studio** (`mnchrmXD/waveform-studio`).

This notebook is optimized for one-click execution:
1. **Automatic Initialization**: Section 1 starts the high-speed Node.js + FFmpeg headless rendering engine on `http://localhost:3000` immediately.
2. **Upload or Paste Payload**: Paste a JSON payload copied from Waveform Studio's **Payload Generator**, upload a `payload.json`, or use the built-in preset.
3. **Hardware Acceleration**: Automatic GPU NVENC detection with multi-threaded fallback and real-time streaming.
4. **Direct Preview & Download**: Instant in-notebook video playback and one-click file download.

## 1. Setup Environment & Launch Server
Prepares dependencies and launches the Waveform Studio headless rendering engine in Colab in the background on `http://localhost:3000`.

In [ ]:
# Install Python utilities
!pip install -q requests tqdm

import os
import sys
import time
import json
import subprocess
import requests
from tqdm import tqdm
from IPython.display import HTML, Video, display

# 1. Ensure repository is available and active directory
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    if not os.path.exists('server.ts') and not os.path.exists('waveform-studio'):
        print("📥 Cloning Waveform Studio repository...")
        !git clone --depth 1 https://github.com/mnchrmXD/waveform-studio.git
        %cd waveform-studio
    elif os.path.exists('waveform-studio') and not os.path.exists('server.ts'):
        %cd waveform-studio

# 2. Fast install Node.js dependencies if not already present
if not os.path.exists('node_modules'):
    print("⚡ Installing Node.js & FFmpeg rendering dependencies...")
    !npm install --prefer-offline --no-audit --no-fund --loglevel=error

# 3. Check if server is already running on port 3000
API_URL = "http://localhost:3000"
server_ready = False

try:
    health_resp = requests.get(f"{API_URL}/api/health", timeout=2)
    if health_resp.status_code == 200:
        server_ready = True
        print("✅ Waveform Studio server is already running!")
except Exception:
    pass

# 4. Launch headless server in the background (HEADLESS_ONLY skips Vite frontend for instant start)
if not server_ready:
    print("🚀 Starting Waveform Studio headless rendering engine...")
    env = os.environ.copy()
    env["HEADLESS_ONLY"] = "true"
    env["NODE_ENV"] = "production"
    
    server_proc = subprocess.Popen(
        ["npx", "tsx", "server.ts"],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    
    # Poll until server is ready
    start_wait = time.time()
    for _ in range(30):
        try:
            health_resp = requests.get(f"{API_URL}/api/health", timeout=1)
            if health_resp.status_code == 200:
                server_ready = True
                print(f"✅ Waveform Studio server ready in {time.time() - start_wait:.1f}s on {API_URL}!")
                print(json.dumps(health_resp.json(), indent=2))
                break
        except Exception:
            time.sleep(0.5)

    if not server_ready:
        print("⚠️ Server initialization taking longer than expected. Check server output if needed.")

## 2. Upload or Configure Payload
You can load your payload into Colab using any of these methods:
- **Method A**: Paste the JSON payload text copied from Waveform Studio's **Payload Generator** into `PASTED_JSON`.
- **Method B**: Upload a `payload.json` file from your computer using the upload cell below.
- **Method C**: Use the default high-performance Python configuration.

In [ ]:
# Optional: Run this cell to upload a 'payload.json' file from your computer
try:
    from google.colab import files
    print("Upload your payload.json (or skip if pasting JSON below):")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.json'):
            os.rename(fn, 'payload.json')
            print("✅ Successfully uploaded and set as payload.json!")
            break
except ImportError:
    pass

In [ ]:
# Paste JSON here if copied from Waveform Studio (leave empty to use Python config or uploaded file)
PASTED_JSON = """"""

payload = None

# 1. Check if payload was pasted
if PASTED_JSON.strip():
    try:
        payload = json.loads(PASTED_JSON)
        print("✅ Loaded payload from PASTED_JSON!")
    except json.JSONDecodeError as err:
        print(f"⚠️ Error parsing PASTED_JSON: {err}")

# 2. Check if payload.json was uploaded
if payload is None and os.path.exists("payload.json"):
    try:
        with open("payload.json", "r") as f:
            payload = json.load(f)
        print("✅ Loaded payload from uploaded payload.json!")
    except Exception as err:
        print(f"⚠️ Error reading payload.json: {err}")

# 3. Fallback to default high-speed Python configuration
if payload is None:
    print("ℹ️ Using default high-performance Python payload configuration:")
    VIDEO_CONFIG = {
        "width": 1280,       # 1280 (720p HD, ultra-fast), 1920 (1080p Full HD), 3840 (4K UHD)
        "height": 720,
        "fps": 30,           # 30 or 60 fps
        "format": "mp4"      # "mp4" (H.264/AAC) or "webm" (VP9/Opus, supports transparent alpha)
    }

    SETTINGS = {
        "style": "mirrored-bars",        # "mirrored-bars", "bars-up", "smooth-wave", "radial", "digital-matrix", "spine", "spectrum-bands"
        "barCount": 80,                  # Bar density (16 to 128)
        "heightScale": 1.2,              # Amplitude multiplier (0.2 to 3.0)
        "smoothing": 0.65,               # Temporal FFT smoothing (0.0 to 1.0)
        "sensitivity": 1.0,              # Volume sensitivity gain (0.2 to 3.0)
        "softKneeCompression": True,     # Soft-knee peak compression
        "backgroundType": "dark-studio", # "dark-studio", "oled-black", "light-canvas", "gradient-mesh", "transparent"
        "enableJoint": True,             # Edge & profile tapering
        "jointWidth": 20,                # Taper transition width (5% to 40%)
        "jointCurve": "smooth",          # "smooth", "linear", "cubic"
        "trackTitle": "Midnight Horizons",
        "artistName": "Waveform Studio",
        "showTrackInfo": True,
        "infoPosition": "top-left",
        "showProfileImage": True,
        "profileImageShape": "circle",
        "profileAudioReactiveScale": True
    }

    THEME = "cyber-cyan"  # "cyber-cyan", "electric-indigo", "sunset-ember", "emerald-mint", "monochrome-luxe", "solar-flare", "nordic-frost"
    AUDIO_URL = "https://cdn.freesound.org/previews/612/612627_11861866-lq.mp3"
    PROFILE_IMAGE_URL = "https://images.unsplash.com/photo-1511671782779-c97d3d27a1d4?w=400&q=80"

    payload = {
        "video": VIDEO_CONFIG,
        "settings": SETTINGS,
        "theme": THEME,
        "audio": AUDIO_URL,
        "profileImage": PROFILE_IMAGE_URL
    }

video_format = payload.get("video", {}).get("format", "mp4")
w = payload.get("video", {}).get("width", 1280)
h = payload.get("video", {}).get("height", 720)
fps = payload.get("video", {}).get("fps", 30)
print(f"Payload ready: {payload.get('settings', {}).get('style', 'default')} style, {w}×{h} @ {fps}fps ({video_format})")

## 3. Render Video via Headless API
Submit the payload to `/api/render-video`. The server analyzes the audio, renders frames with hardware acceleration, encodes them with FFmpeg, and streams the video back in real-time.

In [ ]:
OUTPUT_FILENAME = f"waveform_render.{payload.get('video', {}).get('format', 'mp4')}"
render_url = f"{API_URL}/api/render-video"

print(f"🚀 Submitting render request to {render_url}...")
start_time = time.time()

try:
    response = requests.post(render_url, json=payload, stream=True, timeout=600)
    
    if response.status_code == 200:
        total_size = int(response.headers.get('content-length', 0))
        block_size = 1024 * 1024  # 1MB chunks
        
        with open(OUTPUT_FILENAME, 'wb') as f, tqdm(
            desc=OUTPUT_FILENAME,
            total=total_size,
            unit='iB',
            unit_scale=True,
            unit_divisor=1024,
        ) as bar:
            for chunk in response.iter_content(chunk_size=block_size):
                if chunk:
                    size = f.write(chunk)
                    bar.update(size)
                    
        elapsed = time.time() - start_time
        file_size_mb = os.path.getsize(OUTPUT_FILENAME) / (1024 * 1024)
        print(f"\n🎉 Render successfully completed in {elapsed:.1f} seconds!")
        print(f"📁 Output saved to: {OUTPUT_FILENAME} ({file_size_mb:.2f} MB)")
    else:
        print(f"❌ Render failed with HTTP status {response.status_code}: {response.text}")
except Exception as e:
    print(f"❌ Request failed: {e}")

## 4. Preview & Playback Video
Preview the generated video directly within Colab.

In [ ]:
if os.path.exists(OUTPUT_FILENAME):
    print("Previewing rendered video:")
    display(Video(OUTPUT_FILENAME, embed=True, width=720))
else:
    print(f"File {OUTPUT_FILENAME} does not exist yet. Run Section 3 first.")

## 5. Download Video to Local Machine
Download the finished video directly to your computer.

In [ ]:
try:
    from google.colab import files
    if os.path.exists(OUTPUT_FILENAME):
        print(f"Downloading {OUTPUT_FILENAME}...")
        files.download(OUTPUT_FILENAME)
    else:
        print(f"File {OUTPUT_FILENAME} not found.")
except ImportError:
    print(f"Not running in Google Colab. File is saved at: {os.path.abspath(OUTPUT_FILENAME)}")